# PMIL on Colab

Runs the Colab-adapted PMIL pipeline end to end: patch encoder (`max-mil.py`) → prototype clustering (`cluster.py`) → slide classifier (`pmil.py`).

**Before you hit Runtime → Run all**, fill in the config cell below (Cell 3) with your own paths — your WSIs and `lib/*.ckpt` files are not included in this notebook or the zip, since they're specific to your dataset. Everything else runs as-is.

A timing/smoke-test cell (Cell 5) is included right before the real training run — it runs 1 epoch on a couple of slides and extrapolates how long the full run will actually take on your data and your GPU, so you're not guessing.

If you don't have `lib/*.ckpt` files yet, see the commented-out builder snippet after the config cell.

## 1. Upload the repo zip and unzip it

In [ ]:
from google.colab import files
print("Select PMIL-colab.zip from your computer:")
uploaded = files.upload()

In [ ]:
import zipfile, os

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

%cd /content/PMIL-colab
!ls

## 2. Install dependencies (OpenSlide + openslide-python)

In [ ]:
!bash setup_colab.sh

## 3. Mount Drive and set your paths — EDIT THIS CELL

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- EDIT THESE ----
PROJECT_DIR    = '/content/drive/MyDrive/PMIL'      # anything persistent on your Drive
TRAIN_LIB      = f'{PROJECT_DIR}/lib/train.ckpt'
VAL_LIB        = f'{PROJECT_DIR}/lib/val.ckpt'
TEST_LIB       = f'{PROJECT_DIR}/lib/test.ckpt'
FEAT_DIR       = f'{PROJECT_DIR}/feat'
MIL_MODEL      = f'{PROJECT_DIR}/model/mil.pth'
PMIL_MODEL     = f'{PROJECT_DIR}/model/pmil_model.pth'
GLOBAL_CLUSTER = f'{PROJECT_DIR}/cluster/global_prototypes.npy'
NEPOCHS        = 30          # how many epochs you actually intend to train max-mil.py for
# ---------------------

import os
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(MIL_MODEL), exist_ok=True)
os.makedirs(os.path.dirname(GLOBAL_CLUSTER), exist_ok=True)

for path, label in [(TRAIN_LIB, 'train_lib'), (VAL_LIB, 'val_lib'), (TEST_LIB, 'test_lib')]:
    print(f"{label}: {'FOUND' if os.path.exists(path) else 'MISSING -> ' + path}")

### Don't have `lib/*.ckpt` files yet?

Each one is a `torch.save()`'d dict your own code builds, with at least:

```python
{
    'slides':  [...],   # full path to each WSI, reachable from Colab (e.g. under your mounted Drive)
    'grid':    [...],   # per-slide list of (x, y) patch coordinates
    'targets': [...],   # per-slide label (0/1)
    'level':   0,       # optional, defaults to 0
    'mult':    1,       # optional, defaults to 1
}
```

Uncomment and adapt this if you need a minimal builder:

In [ ]:
# import torch
#
# lib = {
#     'slides':  ['/content/drive/MyDrive/PMIL/wsi/slide_001.tif', ...],
#     'grid':    [[(0, 0), (256, 0), ...], ...],   # one coordinate list per slide, same order as 'slides'
#     'targets': [0, 1, ...],                       # one label per slide, same order as 'slides'
#     'level': 0,
#     'mult': 1,
# }
# torch.save(lib, TRAIN_LIB)

## 4. Time estimate — smoke test before the real run

Runs `max-mil.py` for 1 epoch on just the first couple of slides in `TRAIN_LIB`, times it, then extrapolates to your real slide count and `NEPOCHS`. This is a genuine timed measurement on your own data/GPU, not a guess — but it only covers `max-mil.py` (the dominant cost); `cluster.py` and `pmil.py` are cheap by comparison and not included in the estimate.

In [ ]:
import torch, time, subprocess, os

SMOKE_SLIDES = 2   # bump to 3-5 for a steadier estimate, at the cost of a longer smoke test

full_lib = torch.load(TRAIN_LIB, map_location='cpu')
total_slides = len(full_lib['slides'])
n_smoke = min(SMOKE_SLIDES, total_slides)

smoke_lib = {
    'slides':  full_lib['slides'][:n_smoke],
    'grid':    full_lib['grid'][:n_smoke],
    'targets': full_lib['targets'][:n_smoke],
}
if 'level' in full_lib: smoke_lib['level'] = full_lib['level']
if 'mult' in full_lib:  smoke_lib['mult']  = full_lib['mult']

smoke_lib_path = '/content/smoke_train.ckpt'
torch.save(smoke_lib, smoke_lib_path)

n_patches_smoke = sum(len(g) for g in smoke_lib['grid'])
n_patches_total = sum(len(g) for g in full_lib['grid'])

print(f"Smoke test: {n_smoke}/{total_slides} slides, {n_patches_smoke} patches "
      f"(full dataset: {n_patches_total} patches)")

start = time.time()
result = subprocess.run(
    ['python', 'max-mil.py',
     '--train_lib', smoke_lib_path,
     '--val_lib', smoke_lib_path,
     '--test_lib', smoke_lib_path,
     '--mil_model', MIL_MODEL,
     '--feat_dir', '/content/smoke_feat',
     '--nepochs', '1'],
    capture_output=True, text=True
)
elapsed = time.time() - start

os.makedirs('/content/smoke_feat', exist_ok=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("--- STDERR (smoke run failed, fix this before the real run) ---")
    print(result.stderr[-2000:])
else:
    per_epoch_full = elapsed * (n_patches_total / n_patches_smoke)
    total_estimate_sec = per_epoch_full * NEPOCHS
    print(f"\nSmoke run (1 epoch, {n_smoke} slides): {elapsed:.1f} sec")
    print(f"Extrapolated: ~{per_epoch_full/60:.1f} min/epoch on the full dataset")
    print(f"Extrapolated: ~{total_estimate_sec/3600:.1f} hours for all {NEPOCHS} epochs "
          f"(max-mil.py only -- cluster.py/pmil.py are additional but comparatively cheap)")

## 5. Patch encoder — `max-mil.py`
Skip this cell if you already have a trained `mil.pth` at `MIL_MODEL` and just want to reuse it.

In [ ]:
!python max-mil.py \
    --train_lib "{TRAIN_LIB}" \
    --val_lib "{VAL_LIB}" \
    --test_lib "{TEST_LIB}" \
    --mil_model "{MIL_MODEL}" \
    --feat_dir "{FEAT_DIR}" \
    --nepochs {NEPOCHS} \
    --save_model \
    --save_index \
    --save_feat

## 6. Prototype clustering — `cluster.py`
CPU-only (sklearn Affinity Propagation), needs the per-slide feature files produced above.

In [ ]:
!python cluster.py \
    --feat_dir "{FEAT_DIR}" \
    --lib "{PROJECT_DIR}/lib/select_train.ckpt" \
    --save_path "{GLOBAL_CLUSTER}"

## 7. Final slide classifier — `pmil.py`

In [ ]:
!python pmil.py \
    --train_lib "{TRAIN_LIB}" \
    --val_lib "{VAL_LIB}" \
    --test_lib "{TEST_LIB}" \
    --feat_dir "{FEAT_DIR}" \
    --mil_model "{MIL_MODEL}" \
    --pmil_model "{PMIL_MODEL}" \
    --global_cluster "{GLOBAL_CLUSTER}" \
    --save_model

## Done

Trained checkpoints (`mil.pth`, `pmil_model.pth`), extracted features, and cluster prototypes are all saved under your mounted `PROJECT_DIR` on Drive, so they'll still be there next session — Colab's local `/content` disk is wiped when the runtime resets.

Command-line flags used above are the defaults from the repo's `README.md`; check there (or `--help` on each script) for the full list if you want to tune batch size, epochs, number of prototypes, etc.